In [13]:
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np

In [3]:
path = 'Datasets/'
CHUNK_SIZE = 500_000

In [4]:
pf = pq.ParquetFile(path + "SharedResponses_clean.parquet")
total_rows = pf.metadata.num_rows
total_chunks = (total_rows // CHUNK_SIZE) + 1
print(f"Total rows: {total_rows:,} → {total_chunks} chunks of {CHUNK_SIZE:,}")

Total rows: 62,231,803 → 125 chunks of 500,000


In [5]:
survey = (
    pd.read_csv(path + "SharedResponsesSurvey.csv", low_memory=False,
                usecols=['ExtendedSessionID', 'Review_age', 'Review_gender', 'Review_education',
                        'Review_income', 'Review_political', 'Review_religious'])
    .drop_duplicates(subset='ExtendedSessionID')
)
print(f"Survey sessions loaded: {len(survey):,}")

Survey sessions loaded: 492,845


In [6]:
chunks = []
pbar = tqdm(pf.iter_batches(batch_size=CHUNK_SIZE), total=total_chunks, desc="Loading chunks")
try:
    for batch in pbar:
        chunk = batch.to_pandas()
        if isinstance(chunk['ExtendedSessionID'].dtype, pd.CategoricalDtype):
            chunk['ExtendedSessionID'] = chunk['ExtendedSessionID'].astype(str)
        chunk = chunk.merge(survey, on='ExtendedSessionID', how='left')
        chunks.append(chunk)
finally:
    pbar.close()

print("Concatenating chunks...", end=" ", flush=True)
df = pd.concat(chunks, ignore_index=True)
print("done.")  

print(f"\nFinal shape: {df.shape}")
print(f"Rows with survey data: {df['Review_gender'].notna().sum():,} ({df['Review_gender'].notna().mean()*100:.1f}%)")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(df.head())

Loading chunks: 265it [02:18,  1.91it/s]                         

Concatenating chunks... 

done.

Final shape: (62231803, 40)
Rows with survey data: 10,046,584 (16.1%)
Memory usage: 8.01 GB
               ExtendedSessionID  Intervention  PedPed  Barrier  \
0    32757157_6999801415950060.0         False   False    False   
1  -1613944085_422160228641876.0         False    True    False   
2   1425316635_327833569077076.0         False   False     True   
3  -1683127088_785070916172117.0         False    True    False   
4  1525185249_1436495773909467.0         False   False     True   

   CrossingSignal AttributeLevel ScenarioTypeStrict DefaultChoice  \
0               1            Fit            Fitness           Fit   
1               1         Female             Gender          Male   
2               0            Old                Age         Young   
3               2           More        Utilitarian          More   
4               0            Low      Social Status          High   

  NonDefaultChoice  DefaultChoiceIsOmission  ...  FemaleDoctor  MaleDoctor  \
0    

In [7]:
df = df.dropna(axis=0)
df

,ExtendedSessionID,Intervention,PedPed,Barrier,CrossingSignal,AttributeLevel,ScenarioTypeStrict,DefaultChoice,NonDefaultChoice,DefaultChoiceIsOmission,...,FemaleDoctor,MaleDoctor,Dog,Cat,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious
4,1525185249_1436495773909467.0,False,False,True,0,Low,Social Status,High,Low,False,...,0,0,0,0,26.0,bachelor,male,5000,0.50,0.50
17,-1033736141_5392791780749771.0,False,False,True,0,Male,Gender,Male,Female,True,...,0,0,0,0,14.0,underHigh,female,default,0.50,0.57
31,-972264246_4277296232223713.0,False,False,True,0,Pets,Species,Hoomans,Pets,False,...,0,0,4,1,20.0,high,male,under5000,0.75,0.84
34,-841718081_3084184331213722.0,False,False,False,2,Pets,Species,Hoomans,Pets,False,...,0,0,2,2,19.0,high,male,default,1.00,0.00
40,1247311683_3858850116393528.0,False,True,False,2,Female,Gender,Male,Female,False,...,2,0,0,0,38.0,others,male,above100000,0.68,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62231755,725752586_6904186263183265.0,True,False,False,0,Pets,Species,Hoomans,Pets,True,...,0,0,4,1,22.0,high,male,35000,0.50,0.00
62231759,1479950156_1327764658273678.0,True,True,False,2,More,Utilitarian,More,Less,False,...,2,0,0,0,40.0,college,male,5000,0.76,0.17
62231764,-1785551894_9617570104574178.0,True,False,False,0,Less,Utilitarian,More,Less,True,...,0,0,0,0,16.0,underHigh,male,default,0.78,0.00
62231773,-671133433_3786068032082825.0,True,True,False,1,Less,Utilitarian,More,Less,True,...,0,0,0,0,20.0,college,male,under5000,0.14,0.19


In [ ]:
df_encoded = df.copy()

# Pandas `category` dtype columns → integer codes
pandas_cat_cols = df_encoded.select_dtypes(include='category').columns.tolist()
cat_code_maps = {}

for col in pandas_cat_cols:
    new_col = f"{col}_enc"
    df_encoded[new_col] = df_encoded[col].cat.codes
    cat_code_maps[col] = dict(enumerate(df_encoded[col].cat.categories))
    print(f"  [cat.codes] {col:30s} → {new_col}  "
          f"categories: {list(df_encoded[col].cat.categories)}")

# Ordinal string columns
ordinal_maps = {
    'Review_education': {
        'underhigh':   0,
        'high':        1,
        'vocational':  2,
        'college':     3,
        'bachelor':    4,
        'graduate':    5,
        'others':     -1,
        'default':    -1,
    },
    'Review_income': {
        'under5000':    0,
        '5000':         1,
        '10000':        2,
        'over10000':    3,
        '15000':        4,
        '25000':        5,
        '35000':        6,
        '50000':        7,
        '80000':        8,
        'above100000':  9,
        'default':     -1,
    },
}

for col, mapping in ordinal_maps.items():
    if col not in df_encoded.columns:
        continue
    new_col = f"{col}_enc"
    normalised = df_encoded[col].astype(str).str.strip().str.lower()
    df_encoded[new_col] = normalised.map(mapping).fillna(-1).astype(int)
    n_unmapped = (df_encoded[new_col] == -1).sum()
    print(f"  [ordinal]   {col:30s} → {new_col}  (unmapped rows: {n_unmapped:,})")
    if n_unmapped > 0:
        print(f"              Unseen values: "
              f"{normalised[df_encoded[new_col] == -1].value_counts().head(5).to_dict()}")

# Nominal string columns
nominal_str_cols = ['Review_gender', 'UserCountry3']
label_encoders = {}

for col in nominal_str_cols:
    if col not in df_encoded.columns:
        continue
    new_col = f"{col}_enc"
    series = df_encoded[col].astype(str).str.strip()
    series = series.replace({'': 'MISSING', 'nan': 'MISSING'})
    le = LabelEncoder()
    df_encoded[new_col] = le.fit_transform(series)
    label_encoders[col] = le
    print(f"  [nominal]   {col:30s} → {new_col}  "
          f"({len(le.classes_)} classes: {list(le.classes_[:6])}{'…' if len(le.classes_) > 6 else ''})")

# Clean & standardize continuous numeric features
continuous_cols = ['Review_age', 'Review_political', 'Review_religious']
continuous_cols = [c for c in continuous_cols if c in df_encoded.columns]

valid_ranges = {
    'Review_age':       (0,   120),
    'Review_political': (0.0,   1.0),
    'Review_religious': (0.0,   1.0),
}

if continuous_cols:
    for col in continuous_cols:
        n_inf = np.isinf(df_encoded[col]).sum()
        df_encoded[col] = df_encoded[col].replace([np.inf, -np.inf], np.nan)
        if col in valid_ranges:
            lo, hi = valid_ranges[col]
            n_oob = ((df_encoded[col] < lo) | (df_encoded[col] > hi)).sum()
            df_encoded[col] = df_encoded[col].where(df_encoded[col].between(lo, hi), other=np.nan)
        else:
            n_oob = 0
        median_val = df_encoded[col].median()
        n_nan = df_encoded[col].isna().sum()
        df_encoded[col] = df_encoded[col].fillna(median_val)
        print(f"  [cleaned]   {col:25s}  inf: {n_inf:,}  out-of-range: {n_oob:,}  "
              f"NaN→median({median_val:.4f}): {n_nan:,}")

    scaler = StandardScaler()
    scaled_array = scaler.fit_transform(df_encoded[continuous_cols])
    for i, col in enumerate(continuous_cols):
        df_encoded[f"{col}_scaled"] = scaled_array[:, i]
    print(f"\n  [scaled]    {continuous_cols}  →  _scaled variants (μ≈0, σ≈1)")
    for col, mean, std in zip(continuous_cols, scaler.mean_, scaler.scale_):
        print(f"              {col:25s}  mean={mean:.4f}  std={std:.4f}")

# Drop original categorical columns
cols_to_drop = (
    pandas_cat_cols +
    list(ordinal_maps.keys()) +
    nominal_str_cols +
    continuous_cols
)
cols_to_drop = [c for c in cols_to_drop if c in df_encoded.columns]
df_encoded.drop(columns=cols_to_drop, inplace=True)
print(f"\n  [dropped]   {len(cols_to_drop)} original columns: {cols_to_drop}")

# Summary
enc_cols    = [c for c in df_encoded.columns if c.endswith('_enc')]
scaled_cols = [c for c in df_encoded.columns if c.endswith('_scaled')]

print(f"\n{'─'*60}")
print(f"Original shape   : {df.shape}")
print(f"Final shape      : {df_encoded.shape}")
print(f"New _enc cols    : {enc_cols}")
print(f"New _scaled cols : {scaled_cols}")
print(f"Memory usage     : {df_encoded.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"\n{df_encoded.dtypes}")
print(f"\nEncoded sample:")
print(df_encoded[enc_cols + scaled_cols].head(3).to_string())

  [cat.codes] AttributeLevel                 → AttributeLevel_enc  categories: ['Fat', 'Female', 'Fit', 'High', 'Hoomans', 'Less', 'Low', 'Male', 'More', 'Old', 'Pets', 'Young']
  [cat.codes] ScenarioTypeStrict             → ScenarioTypeStrict_enc  categories: ['Age', 'Fitness', 'Gender', 'Random', 'Social Status', 'Species', 'Utilitarian']
  [cat.codes] DefaultChoice                  → DefaultChoice_enc  categories: ['Fit', 'High', 'Hoomans', 'Male', 'More', 'Young']
  [cat.codes] NonDefaultChoice               → NonDefaultChoice_enc  categories: ['Fat', 'Female', 'Less', 'Low', 'Old', 'Pets']
  [ordinal]   Review_education               → Review_education_enc  (unmapped rows: 817,646)
              Unseen values: {'others': 532830, 'default': 284816}
  [ordinal]   Review_income                  → Review_income_enc  (unmapped rows: 1,493,600)
              Unseen values: {'default': 1493600}
  [nominal]   Review_gender                  → Review_gender_enc  (5 classes: ['apache helicop

In [29]:
df_encoded.to_csv(path + "SharedResponses_combined.csv", index=False)

In [27]:
df_encoded.info()

<class 'pandas.DataFrame'>
Index: 8834024 entries, 4 to 62231784
Data columns (total 40 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   ExtendedSessionID        str    
 1   Intervention             bool   
 2   PedPed                   bool   
 3   Barrier                  bool   
 4   CrossingSignal           uint8  
 5   DefaultChoiceIsOmission  bool   
 6   NumberOfCharacters       uint8  
 7   DiffNumberOFCharacters   int8   
 8   Saved                    bool   
 9   Man                      uint8  
 10  Woman                    uint8  
 11  Pregnant                 uint8  
 12  Stroller                 uint8  
 13  OldMan                   uint8  
 14  OldWoman                 uint8  
 15  Boy                      uint8  
 16  Girl                     uint8  
 17  Homeless                 uint8  
 18  LargeWoman               uint8  
 19  LargeMan                 uint8  
 20  Criminal                 uint8  
 21  MaleExecutive          

In [30]:
df_encoded.head()

,ExtendedSessionID,Intervention,PedPed,Barrier,CrossingSignal,DefaultChoiceIsOmission,NumberOfCharacters,DiffNumberOFCharacters,Saved,Man,...,ScenarioTypeStrict_enc,DefaultChoice_enc,NonDefaultChoice_enc,Review_education_enc,Review_income_enc,Review_gender_enc,UserCountry3_enc,Review_age_scaled,Review_political_scaled,Review_religious_scaled
4,1525185249_1436495773909467.0,False,False,True,0,False,2,0,False,0,...,4,1,3,4,1,3,167,0.160363,-0.404355,0.717880
17,-1033736141_5392791780749771.0,False,False,True,0,True,1,0,False,0,...,2,3,1,0,-1,2,29,-0.964923,-0.404355,0.940279
31,-972264246_4277296232223713.0,False,False,True,0,False,5,0,False,0,...,5,2,5,1,0,3,29,-0.402280,0.530110,1.798104
34,-841718081_3084184331213722.0,False,False,False,2,False,4,0,False,0,...,5,2,5,1,-1,3,29,-0.496054,1.464575,-0.870683
40,1247311683_3858850116393528.0,False,True,False,2,False,2,0,True,0,...,2,3,1,-1,9,3,33,1.285650,0.268459,-0.870683
